# SIESTA single-elements dataset pipeline and validation

This notebook documents the end-to-end pipeline used to create the public dataset and shows a compact set of validation checks.

## Pipeline overview

1. Generate perturbed structures and SIESTA input files with `make_single_elements.py`
2. Run SIESTA calculations with `run_siesta.py`
3. Build the dataset index with `build_index_siesta.py`
4. Slim the dataset for public release with `slim_public_core.py`
5. Validate file completeness, index consistency, NetCDF readability, and convergence


## 1) Example commands for the full pipeline

Edit the paths below to match your machine.


In [ ]:
# Example shell commands (not executed automatically here)
pipeline_commands = r'''
# 1. Generate SIESTA inputs
python 1.make_single_elements.py \
  --out-root raw_single_elements \
  --seed 0 \
  --n-per-element 11 \
  --n-disp 3 \
  --n-iso 3 \
  --n-aniso 2 \
  --n-shear 1 \
  --n-combo 1

# 2. Run SIESTA
python 2.run_siesta.py \
  --raw-root raw_single_elements \
  --pseudo-dir pseudos \
  --jobs 16

# 3. Build index
python 3.build_index_siesta.py raw_single_elements \
  --include-extra \
  --out index_full.jsonl \
  --parquet index_full.parquet

# 4. Build public-core release
python 4.slim_public_core.py \
  --root raw_single_elements \
  --index index_full.parquet \
  --out public_core \
  --mode copy
'''
print(pipeline_commands)


## 2) Set dataset root

Point this notebook to the `public_core` directory that contains `index_core.parquet`, `index_core.jsonl`, and `data/`.


In [ ]:
from pathlib import Path
import json
import statistics
from collections import Counter

try:
    import pandas as pd
except Exception:
    pd = None

PUBLIC_CORE = Path("public_core")  # change if needed
INDEX_PARQUET = PUBLIC_CORE / "index_core.parquet"
INDEX_JSONL = PUBLIC_CORE / "index_core.jsonl"
DATA_ROOT = PUBLIC_CORE / "data"

print("PUBLIC_CORE =", PUBLIC_CORE.resolve())
print("Index parquet exists:", INDEX_PARQUET.exists())
print("Index jsonl exists:", INDEX_JSONL.exists())
print("Data root exists:", DATA_ROOT.exists())


## 3) Load the index

This cell loads the index from Parquet if available; otherwise it falls back to JSONL.


In [ ]:
if INDEX_PARQUET.exists() and pd is not None:
    df = pd.read_parquet(INDEX_PARQUET)
    rows = df.to_dict(orient="records")
    print("Loaded Parquet index with", len(df), "rows")
else:
    rows = [json.loads(line) for line in INDEX_JSONL.read_text().splitlines() if line.strip()]
    df = pd.DataFrame(rows) if pd is not None else None
    print("Loaded JSONL index with", len(rows), "rows")

rows[:2] if rows else []


## 4) Basic integrity checks

These checks reproduce the ones used during dataset validation:

- total number of rows
- element distribution
- missing directories
- missing `Rho.grid.nc`
- missing `ElectrostaticPotential.grid.nc`


In [ ]:
calc_dirs = [r["calc_dir"] for r in rows]
elements = Counter(r["element"] for r in rows)

missing_dirs = []
missing_rho = []
missing_vh = []

for r in rows:
    d = DATA_ROOT / r["calc_dir"]
    if not d.exists():
        missing_dirs.append(r["calc_dir"])
        continue
    if not (d / "Rho.grid.nc").exists():
        missing_rho.append(r["calc_dir"])
    if not (d / "ElectrostaticPotential.grid.nc").exists():
        missing_vh.append(r["calc_dir"])

print("Total rows:", len(rows))
print("Element distribution:", elements)
print("Missing directories:", len(missing_dirs))
print("Missing Rho.grid.nc:", len(missing_rho))
print("Missing ElectrostaticPotential.grid.nc:", len(missing_vh))
print("Examples:", (missing_dirs + missing_rho + missing_vh)[:10])


## 5) Index-to-file consistency

Checks that every indexed calculation points to existing core grid files.


In [ ]:
broken = []
for r in rows:
    d = DATA_ROOT / r["calc_dir"]
    for fname in ["Rho.grid.nc", "ElectrostaticPotential.grid.nc"]:
        if not (d / fname).exists():
            broken.append((r["calc_dir"], fname))

print("Broken index entries:", len(broken))
print(broken[:10])


## 6) NetCDF readability spot check

Randomly samples a few `Rho.grid.nc` files and checks that they open successfully.


In [ ]:
import random
from netCDF4 import Dataset

rho_files = list(DATA_ROOT.glob("*/*/Rho.grid.nc"))
sample = random.sample(rho_files, min(5, len(rho_files)))

for f in sample:
    ds = Dataset(f)
    print(f, "OK | variables:", list(ds.variables.keys())[:5])
    ds.close()


## 7) File size consistency

A quick outlier screen for the density grids.


In [ ]:
sizes = [f.stat().st_size for f in DATA_ROOT.glob("*/*/Rho.grid.nc")]
print("Count:", len(sizes))
print("Min:", min(sizes) if sizes else None)
print("Max:", max(sizes) if sizes else None)
print("Median:", statistics.median(sizes) if sizes else None)


## 8) SCF convergence check

Uses the `scf_converged` flag stored in the index.


In [ ]:
non_converged = [r["calc_dir"] for r in rows if not r.get("scf_converged", True)]
print("Non-converged:", len(non_converged))
print(non_converged[:10])


## 9) Ordering sanity check

Prints the first few calculation directories from the index.


In [ ]:
print("First 5 calc_dirs:", calc_dirs[:5])

## 10) Summary

A release is considered ready when all of the following hold:

- `Missing directories == 0`
- `Missing Rho.grid.nc == 0`
- `Missing ElectrostaticPotential.grid.nc == 0`
- `Broken index entries == 0`
- `Non-converged == 0`
- NetCDF files are readable
- No strong size outliers are detected
